# Credit card default risk: analysis walk-through

This notebook tells the story; the work happens in `src/credit`. Run `make run` first so the feature table, the out-of-fold predictions and `results/metrics.json` exist.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from credit import decision, evaluate
from credit.config import load_config
from credit.features import FEATURES

cfg = load_config(ROOT / "config.yaml")
table = pd.read_parquet(ROOT / cfg["data"]["table"])
oof = pd.read_parquet(ROOT / cfg["data"]["predictions"])
m = json.loads((ROOT / "results" / "metrics.json").read_text())
print(f"{len(table):,} accounts, {len(FEATURES)} model inputs, default rate {table['default'].mean():.1%}")

29,965 accounts, 22 model inputs, default rate 22.1%


## 1. The data

Repayment status dominates: the default rate climbs steeply once an account is even one month late.

In [2]:
(table.groupby("pay_status_1")["default"].agg(["size", "mean"])
 .rename(columns={"size": "accounts", "mean": "default rate"}).round(3))

,accounts,default rate
pay_status_1,,
-2,2750,0.132
-1,5682,0.168
0,14737,0.128
1,3667,0.340
2,2666,0.691
3,322,0.758
4,76,0.684
5,26,0.500
6,11,0.545


In [3]:
pd.DataFrame(m["data"]["cleaning"])

,step,rows_before,dropped,rows_after,changed
0,"Fold undocumented EDUCATION codes 0, 5, 6 into...",30000,0,30000,345
1,Fold undocumented MARRIAGE code 0 into 3 (other),30000,0,30000,54
2,Drop accounts identical on every field except ...,30000,35,29965,0


## 2. Model comparison (out-of-fold)

Every prediction below comes from a model that never saw the account. The bootstrap CIs are paired across models.

In [4]:
rows = {k: {"ROC-AUC": v["roc_auc"], "PR-AUC": v["pr_auc"], "KS": v["ks"], "Brier": v["brier"],
            "calib. error": v["ece"]} for k, v in m["models"].items()}
pd.DataFrame(rows).T.round(4)

,ROC-AUC,PR-AUC,KS,Brier,calib. error
baseline,0.5000,0.2213,0.0000,0.1723,0.0000
scorecard,0.7717,0.5344,0.4168,0.1363,0.0125
logistic,0.7583,0.5157,0.4112,0.1396,0.0077
xgboost,0.7866,0.5623,0.4362,0.1332,0.0051
xgboost_isotonic,0.7865,0.5617,0.4361,0.1333,0.0053


In [5]:
y = oof["default"].to_numpy()
for name in ["scorecard", "logistic", "xgboost", "xgboost_isotonic"]:
    print(f"{name:18s} AUC {evaluate.auc(y, oof[name].to_numpy()):.3f}")
print("XGBoost minus scorecard AUC (paired 95% CI):",
      np.round(m["comparisons"]["xgboost_vs_scorecard_auc"]["ci"], 3))

scorecard          AUC 0.772
logistic           AUC 0.758
xgboost            AUC 0.787
xgboost_isotonic   AUC 0.787
XGBoost minus scorecard AUC (paired 95% CI): [0.012 0.018]


## 3. Calibration

The cut-off rule assumes the PDs are calibrated, so check predicted vs observed by decile.

In [6]:
evaluate.calibration_curve(y, oof["xgboost_isotonic"].to_numpy(), 10).round(3)

,predicted,observed,n
0,0.041,0.045,2997
1,0.066,0.059,2996
2,0.090,0.095,2997
3,0.116,0.109,2996
4,0.142,0.133,2997
5,0.170,0.170,2996
6,0.201,0.193,2996
7,0.266,0.269,2997
8,0.435,0.433,2996
9,0.700,0.707,2997


## 4. Choosing the cut-off

Approve while PD × LGD < (1 − PD) × margin. With LGD 60% and margin 18% the theoretical cut-off is 0.18 / 0.78 ≈ 0.23. The sweep below uses the realised outcomes.

In [7]:
dc = cfg["decision"]
grid = np.round(np.arange(0.05, 0.6, 0.05), 2)
exposure = table["exposure"].to_numpy(dtype=float)
sweep = decision.sweep(y, oof["xgboost_isotonic"].to_numpy(), exposure, grid, dc["lgd"], dc["margin"])
(sweep[["threshold", "approval_rate", "per_1000"]]
 .assign(per_1000=lambda d: (d.per_1000 / 1e6).round(2))
 .rename(columns={"per_1000": "cost per 1,000 accounts (NT$ m)"}))

,threshold,approval_rate,"cost per 1,000 accounts (NT$ m)"
0,0.05,0.078191,6.48
1,0.10,0.291173,4.87
2,0.15,0.478058,4.00
3,0.20,0.646588,3.69
4,0.25,0.742433,3.58
5,0.30,0.779276,3.65
6,0.35,0.811180,3.70
7,0.40,0.840013,3.76
8,0.45,0.860070,3.85
9,0.50,0.877223,3.95


In [8]:
pd.DataFrame(m["decision"]["sensitivity"]).round(3)

,lgd,threshold,analytic,approval_rate,saving_per_1000,saving_share,regret_of_base_threshold
0,0.4,0.32,0.310,0.792,1658048.352,0.385,13076.950
1,0.6,0.25,0.231,0.742,2880398.046,0.446,0.000
2,0.8,0.17,0.184,0.552,4221265.363,0.490,105440.672


XGBoost ranks better, but at each model's best cut-off the scorecard costs the same within bootstrap noise:

In [9]:
m["decision"]["xgboost_minus_scorecard_cost"]

{'mean': 3039.0792844986063,
 'ci': [-51905.14395127669, 52510.34907391942],
 'share_a_cheaper': 0.459}

## 5. Segments and fairness

Sex is not a model input; outcomes are compared by sex after the fact.

In [10]:
cols = ["segment", "level", "n", "default_rate", "mean_pd", "roc_auc", "approval_rate"]
pd.DataFrame(m["segments"])[cols].round(3)

,segment,level,n,default_rate,mean_pd,roc_auc,approval_rate
0,age_band,20-29,9603,0.229,0.230,0.788,0.732
1,age_band,30-39,11226,0.203,0.204,0.785,0.772
2,age_band,40-49,6456,0.229,0.229,0.787,0.731
3,age_band,50-79,2680,0.253,0.256,0.776,0.684
4,education_label,graduate school,10563,0.192,0.194,0.791,0.788
5,education_label,high school,4915,0.252,0.251,0.778,0.684
6,education_label,other,468,0.071,0.119,0.607,0.919
7,education_label,university,14019,0.237,0.237,0.783,0.723
8,limit_band,NT$0k-50k,7673,0.318,0.318,0.755,0.573
9,limit_band,NT$50k-140k,7607,0.247,0.246,0.782,0.708


In [11]:
pd.DataFrame(m["fairness"]).round(3)

,group,n,default_rate,mean_pd,approval_rate,rejected_good_rate,approved_bad_rate
0,female,18091,0.208,0.214,0.759,0.151,0.415
1,male,11874,0.242,0.236,0.717,0.178,0.389


## 6. The scorecard

Points for the strongest feature; the full table is in `results/scorecard_points.csv`.

In [12]:
pts = pd.read_csv(ROOT / "results" / "scorecard_points.csv")
pts[pts["feature"] == "pay_status_1"]

,feature,bin,accounts,default_rate,woe,points
6,pay_status_1,"(-inf, -1]",8432,0.156309,0.427507,37
7,pay_status_1,"(-1, 0]",14737,0.128113,0.659388,43
8,pay_status_1,"(0, 1]",3667,0.340333,-0.596516,11
9,pay_status_1,"(1, inf]",3129,0.695430,-2.083468,-28


## Takeaways

- Payment history carries most of the signal; utilisation and payment ratios add a little.
- XGBoost wins on ranking by about 0.015 AUC but not on the cost of the decision, so the readable scorecard is the better deployment.
- The cost curve is flat near its minimum, so the rule is robust to the LGD assumption.